# Data Analyst Skills — Python for Data Analysis
This notebook covers, in order:
1. Python fundamentals (variables, functions, loops, lists/dicts, file handling)
2. Loading & cleaning a CSV dataset with **pandas** / **numpy**
3. KPI calculation
4. Descriptive statistics, correlation, and trend analysis (EDA)
5. Visualizations with **matplotlib**

Dataset: `../data/sales_data_raw.csv` (synthetic retail sales data, intentionally messy)

## 1. Python Fundamentals
Quick, self-contained examples of the core building blocks before we bring in pandas.

In [1]:
# Variables & basic types
company_name = "Acme Retail Co."
fiscal_year = 2025
target_revenue = 7_000_000.0
is_public_company = False

print(company_name, fiscal_year, target_revenue, is_public_company)

Acme Retail Co. 2025 7000000.0 False


In [2]:
# Functions
def profit_margin(sales: float, profit: float) -> float:
    """Return profit as a percentage of sales. Returns 0 if sales is 0."""
    return (profit / sales * 100) if sales else 0.0

print(f"Margin example: {profit_margin(2000, 550):.1f}%")

Margin example: 27.5%


In [3]:
# Loops + Lists & Dictionaries
regional_sales = [
    {"region": "North", "sales": 1_611_825},
    {"region": "South", "sales": 1_879_321},
    {"region": "East",  "sales": 1_832_863},
    {"region": "West",  "sales": 1_751_218},
]

total = 0
for row in regional_sales:
    total += row["sales"]
    print(f"{row['region']:<6} -> ${row['sales']:,}")

print(f"\nTotal across regions: ${total:,}")

North  -> $1,611,825
South  -> $1,879,321
East   -> $1,832,863
West   -> $1,751,218

Total across regions: $7,075,227


In [4]:
# File handling — read/write a small text summary (plain Python, no pandas)
summary_path = "../eda/quick_region_summary.txt"

with open(summary_path, "w") as f:
    for row in regional_sales:
        f.write(f"{row['region']}: ${row['sales']:,}\n")

with open(summary_path, "r") as f:
    print(f.read())

North: $1,611,825
South: $1,879,321
East: $1,832,863
West: $1,751,218



## 2. Load the Raw Dataset

In [5]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe backend; drop this line in a normal Jupyter session
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.width", 120)

RAW_PATH = "../data/sales_data_raw.csv"
CLEAN_PATH = "../data/sales_data_cleaned.csv"
VIZ_DIR = "../visuals"

df = pd.read_csv(RAW_PATH)
print("Shape:", df.shape)
df.head()

Shape: (2025, 14)


,OrderID,OrderDate,ShipDate,ShipMode,CustomerID,CustomerName,Region,Category,Product,Quantity,UnitPrice,Discount,Sales,Profit
0,ORD-10535,2024-04-12,2024-04-18,Second Class,CUST-0008,Customer 8,East,Furniture,Chair,4.0,537.65,0.00,2150.60,783.99
1,ORD-11590,2024-01-03,2024-01-09,Second Class,CUST-0187,Customer 187,North,Office Supplies,Stapler,4.0,753.55,0.00,3014.20,985.55
2,ORD-11372,2024-08-06,2024-08-07,Second Class,CUST-0015,Customer 15,East,Furniture,Table Lamp,1.0,1169.19,0.15,993.81,142.96
3,ORD-10343,2024-04-07,2024-04-12,Second Class,CUST-0178,Customer 178,North,Furniture,Desk,3.0,1183.31,0.00,3549.93,1528.81
4,ORD-11320,2025-06-16,2025-06-21,First Class,CUST-0004,Customer 4,South,Technology,Laptop,6.0,934.04,0.10,5043.82,719.20


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2025 entries, 0 to 2024
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   OrderID       2025 non-null   str    
 1   OrderDate     2025 non-null   str    
 2   ShipDate      2025 non-null   str    
 3   ShipMode      1985 non-null   str    
 4   CustomerID    2025 non-null   str    
 5   CustomerName  2025 non-null   str    
 6   Region        2005 non-null   str    
 7   Category      2025 non-null   str    
 8   Product       2025 non-null   str    
 9   Quantity      1985 non-null   float64
 10  UnitPrice     1995 non-null   float64
 11  Discount      1965 non-null   float64
 12  Sales         2025 non-null   float64
 13  Profit        2005 non-null   float64
dtypes: float64(5), str(9)
memory usage: 221.6 KB


In [7]:
# Missing values per column
df.isna().sum()

OrderID          0
OrderDate        0
ShipDate         0
ShipMode        40
CustomerID       0
CustomerName     0
Region          20
Category         0
Product          0
Quantity        40
UnitPrice       30
Discount        60
Sales            0
Profit          20
dtype: int64

## 3. Data Cleaning

In [8]:
# Remove exact duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows")

Removed 7 duplicate rows


In [9]:
# Standardize text columns: strip whitespace, fix casing
text_cols = ["CustomerName", "Region", "Category", "Product", "ShipMode"]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()
df["Region"] = df["Region"].str.title()
df["Category"] = df["Category"].str.title()
df.loc[df["Region"] == "Nan", "Region"] = np.nan

In [10]:
# Parse mixed-format dates (both YYYY-MM-DD and DD/MM/YYYY appear in the raw export)
def parse_mixed_date(value):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y"):
        try:
            return pd.to_datetime(value, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

df["OrderDate"] = df["OrderDate"].apply(parse_mixed_date)
df["ShipDate"] = pd.to_datetime(df["ShipDate"], errors="coerce")

In [11]:
# Handle missing values
# Numeric columns -> median within the same Category (robust + explainable)
for col in ["Quantity", "UnitPrice", "Discount", "Profit"]:
    df[col] = df.groupby("Category")[col].transform(lambda s: s.fillna(s.median()))

# Categorical columns -> mode (most frequent value)
for col in ["Region", "ShipMode"]:
    df[col] = df[col].fillna(df[col].mode().iloc[0])

df.isna().sum()

OrderID         0
OrderDate       0
ShipDate        0
ShipMode        0
CustomerID      0
CustomerName    0
Region          0
Category        0
Product         0
Quantity        0
UnitPrice       0
Discount        0
Sales           0
Profit          0
dtype: int64

In [12]:
# Fix invalid values: quantities must be positive
invalid_qty = (df["Quantity"] <= 0).sum()
df["Quantity"] = df["Quantity"].abs().replace(0, 1)
print(f"Fixed {invalid_qty} rows with non-positive Quantity")

Fixed 4 rows with non-positive Quantity


In [13]:
# Outlier detection with the IQR method — flag, don\'t silently delete
def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

low, high = iqr_bounds(df["Sales"])
df["Sales_Outlier"] = ~df["Sales"].between(low, high)
low_q, high_q = iqr_bounds(df["Quantity"])
df["Quantity_Outlier"] = ~df["Quantity"].between(low_q, high_q)
print("Sales outliers:", df["Sales_Outlier"].sum(), "| Quantity outliers:", df["Quantity_Outlier"].sum())

Sales outliers: 47 | Quantity outliers: 3


In [14]:
# Data validation: does Sales line up with Quantity * UnitPrice * (1 - Discount)?
expected_sales = (df["Quantity"] * df["UnitPrice"] * (1 - df["Discount"])).round(2)
mismatch = (df["Sales"] - expected_sales).abs() > (0.25 * expected_sales.clip(lower=1))
df["Sales_Validation_Flag"] = mismatch
print(f"Rows flagged for Sales/inputs mismatch: {mismatch.sum()}")

Rows flagged for Sales/inputs mismatch: 37


In [15]:
# Derived columns for reporting
df["OrderMonth"] = df["OrderDate"].dt.to_period("M").astype(str)
df["ShippingDays"] = (df["ShipDate"] - df["OrderDate"]).dt.days
df["ProfitMargin"] = (df["Profit"] / df["Sales"]).round(4)

df.to_csv(CLEAN_PATH, index=False)
print(f"Cleaned dataset saved -> {CLEAN_PATH}  shape={df.shape}")
df.head()

Cleaned dataset saved -> ../data/sales_data_cleaned.csv  shape=(2018, 20)


,OrderID,OrderDate,ShipDate,ShipMode,CustomerID,CustomerName,Region,Category,Product,Quantity,UnitPrice,Discount,Sales,Profit,Sales_Outlier,Quantity_Outlier,Sales_Validation_Flag,OrderMonth,ShippingDays,ProfitMargin
0,ORD-10535,2024-04-12,2024-04-18,Second Class,CUST-0008,Customer 8,East,Furniture,Chair,4.0,537.65,0.00,2150.60,783.99,False,False,False,2024-04,6,0.3645
1,ORD-11590,2024-01-03,2024-01-09,Second Class,CUST-0187,Customer 187,North,Office Supplies,Stapler,4.0,753.55,0.00,3014.20,985.55,False,False,False,2024-01,6,0.3270
2,ORD-11372,2024-08-06,2024-08-07,Second Class,CUST-0015,Customer 15,East,Furniture,Table Lamp,1.0,1169.19,0.15,993.81,142.96,False,False,False,2024-08,1,0.1439
3,ORD-10343,2024-04-07,2024-04-12,Second Class,CUST-0178,Customer 178,North,Furniture,Desk,3.0,1183.31,0.00,3549.93,1528.81,False,False,False,2024-04,5,0.4307
4,ORD-11320,2025-06-16,2025-06-21,First Class,CUST-0004,Customer 4,South,Technology,Laptop,6.0,934.04,0.10,5043.82,719.20,False,False,False,2025-06,5,0.1426


## 4. KPI Calculation

In [16]:
kpis = {
    "Total Revenue": df["Sales"].sum(),
    "Total Profit": df["Profit"].sum(),
    "Overall Profit Margin %": 100 * df["Profit"].sum() / df["Sales"].sum(),
    "Total Orders": df["OrderID"].nunique(),
    "Total Units Sold": df["Quantity"].sum(),
    "Average Order Value": df.groupby("OrderID")["Sales"].sum().mean(),
    "Average Discount %": 100 * df["Discount"].mean(),
    "Average Shipping Days": df["ShippingDays"].mean(),
}
for k, v in kpis.items():
    print(f"{k:<28}: {v:,.2f}")

Total Revenue               : 7,075,226.37
Total Profit                : 1,950,391.02
Overall Profit Margin %     : 27.57
Total Orders                : 2,000.00
Total Units Sold            : 12,849.00
Average Order Value         : 3,537.61
Average Discount %          : 5.38
Average Shipping Days       : 3.96


## 5. Descriptive Statistics, Correlation & Trend Analysis

In [17]:
stats = df[["Sales", "Profit", "Quantity", "Discount"]].agg(["mean", "median", "std"])
mode_row = df[["Sales", "Profit", "Quantity", "Discount"]].mode().iloc[0]
print(stats)
print("\nMode:\n", mode_row)

              Sales       Profit   Quantity  Discount
mean    3506.058658   966.497039   6.367195  0.053840
median  2546.660000   634.525000   6.000000  0.000000
std     4947.406435  1007.142653  11.116477  0.083291

Mode:
 Sales       204.770
Profit      646.945
Quantity      6.000
Discount      0.000
Name: 0, dtype: float64


In [18]:
corr = df[["Sales", "Profit", "Quantity", "Discount", "UnitPrice"]].corr()
corr.round(3)

,Sales,Profit,Quantity,Discount,UnitPrice
Sales,1.000,0.541,0.096,-0.071,0.400
Profit,0.541,1.000,0.134,-0.301,0.537
Quantity,0.096,0.134,1.000,-0.028,-0.005
Discount,-0.071,-0.301,-0.028,1.000,-0.017
UnitPrice,0.400,0.537,-0.005,-0.017,1.000


In [19]:
monthly = df.groupby("OrderMonth")["Sales"].sum().sort_index()
by_region = df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
by_category = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
top_products = df.groupby("Product")["Sales"].sum().sort_values(ascending=False).head(10)

print(by_region, "\n")
print(by_category, "\n")
print(top_products)

Region
South    1.879321e+06
East     1.832863e+06
West     1.751218e+06
North    1.611825e+06
Name: Sales, dtype: float64 

Category
Office Supplies    2.466267e+06
Technology         2.334291e+06
Furniture          2.274668e+06
Name: Sales, dtype: float64 

Product
Desk              690035.492594
Paper             689454.475682
Monitor           677731.150000
Pens (Box)        661403.460000
Binder            570980.960000
Bookcase          564130.533054
Laptop            563585.360000
Printer           551454.970000
Stapler           544428.140000
Wireless Mouse    541519.640000
Name: Sales, dtype: float64


## 6. Visualizations

In [20]:
fig, ax = plt.subplots(figsize=(9, 4.5))
monthly.plot(kind="line", marker="o", ax=ax, color="#2563eb")
ax.set_title("Monthly Sales Trend")
ax.set_ylabel("Sales ($)")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/01_monthly_sales_trend.png", dpi=150)
plt.show()

In [21]:
fig, ax = plt.subplots(figsize=(6, 4.5))
by_region.plot(kind="bar", ax=ax, color="#16a34a")
ax.set_title("Total Sales by Region")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/02_sales_by_region.png", dpi=150)
plt.show()

In [22]:
fig, ax = plt.subplots(figsize=(6, 6))
by_category.plot(kind="pie", ax=ax, autopct="%1.1f%%", ylabel="", colors=["#2563eb", "#16a34a", "#f59e0b"])
ax.set_title("Sales Share by Category")
plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/03_sales_by_category.png", dpi=150)
plt.show()

In [23]:
fig, ax = plt.subplots(figsize=(8, 5))
top_products.sort_values().plot(kind="barh", ax=ax, color="#7c3aed")
ax.set_title("Top 10 Products by Sales")
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/04_top_products.png", dpi=150)
plt.show()

In [24]:
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns))); ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Correlation Matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/05_correlation_heatmap.png", dpi=150)
plt.show()

In [25]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.boxplot(df["Sales"], vert=False)
ax.set_title("Sales Distribution & Outliers")
ax.set_xlabel("Sales ($)")
plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/06_sales_outliers_boxplot.png", dpi=150)
plt.show()

## 7. Key Takeaways
See `../eda/eda_report.md` for the full written report. Headline findings:
- Sales distribution is right-skewed (mean > median) — a handful of large orders drive a
  disproportionate share of revenue.
- Discount correlates negatively with Profit (-0.30) and more strongly with Profit
  Margin (-0.55) — discounting is eroding margin faster than it drives volume.
- South and East regions lead revenue; North lags by ~18%.
- Category mix is well balanced across Furniture, Technology, and Office Supplies.